# Temporal fraud anomaly detection



In [1]:
!pip install -q kagglehub tabpfn mlflow

import os
import random
from getpass import getpass

import numpy as np
import pandas as pd
import tensorflow as tf
import torch
import kagglehub
import mlflow
import mlflow.sklearn
import mlflow.keras
from tabpfn import TabPFNClassifier
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

LABEL_COL = "is_fraud"
N_TRAIN_NORMAL = 100_000
VAL_FRAC = 0.15
N_TEST_NORMAL = 20_000
FPR_TARGET = 0.05
ADD_CARD_FEATURES = True
EPOCHS = 30
K_NORMAL = 6000
TABPFN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Load data

In [2]:
data_dir = kagglehub.dataset_download("kartik2112/fraud-detection")

train_raw = pd.read_csv(os.path.join(data_dir, "fraudTrain.csv"), index_col=0)
test_raw = pd.read_csv(os.path.join(data_dir, "fraudTest.csv"), index_col=0)
train_raw["split"] = "train"
test_raw["split"] = "test"

df = pd.concat([train_raw, test_raw], ignore_index=True)
del train_raw, test_raw

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Using Colab cache for faster access to the 'fraud-detection' dataset.
Dataset shape: (1852394, 23)
Columns: ['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'split']


,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,split
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,train
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,train
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,train
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0,train
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0,train


In [3]:
print(df.groupby("split")[LABEL_COL].agg(rows="size", fraud="sum", fraud_rate="mean"))

print("Fraud rate by category:")
display(
    df.groupby("category")[LABEL_COL]
    .agg(rows="size", fraud="sum", fraud_rate="mean")
    .sort_values("fraud_rate", ascending=False)
)

          rows  fraud  fraud_rate
split                            
test    555719   2145    0.003860
train  1296675   7506    0.005789
Fraud rate by category:


,rows,fraud,fraud_rate
category,,,
shopping_net,139322,2219,0.015927
misc_net,90654,1182,0.013039
grocery_pos,176191,2228,0.012645
shopping_pos,166463,1056,0.006344
gas_transport,188029,772,0.004106
misc_pos,114229,322,0.002819
grocery_net,64878,175,0.002697
travel,57956,156,0.002692
personal_care,130085,290,0.002229


## 2. Feature engineering

The dataset does not have a `fraud_type` column, so fraud is treated as one class.

In [4]:
df["ts"] = pd.to_datetime(df["trans_date_trans_time"])
df = df.sort_values("ts", kind="stable").reset_index(drop=True)

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

df["hour"] = df["ts"].dt.hour
df["dayofweek"] = df["ts"].dt.dayofweek
df["month"] = df["ts"].dt.month
df["age"] = (df["ts"] - pd.to_datetime(df["dob"])).dt.days / 365.25
df["log_amt"] = np.log1p(df["amt"])
df["log_city_pop"] = np.log1p(df["city_pop"])
df["distance_km"] = haversine_km(df["lat"], df["long"], df["merch_lat"], df["merch_long"])

card_feats = []
if ADD_CARD_FEATURES:
    g = df.groupby("cc_num")
    gap = g["unix_time"].diff()
    df["log_secs_since_prev"] = np.log1p(gap).fillna(np.log1p(gap.median()))

    n_prev = g.cumcount()
    past_mean = ((g["amt"].cumsum() - df["amt"]) / n_prev.replace(0, np.nan)).clip(lower=0)
    df["log_amt_vs_card_mean"] = (np.log1p(df["amt"]) - np.log1p(past_mean)).fillna(0.0)
    card_feats = ["log_secs_since_prev", "log_amt_vs_card_mean"]

categorical_cols = ["category", "gender", "state"]
numerical_cols = [
    "log_amt", "distance_km", "age", "log_city_pop",
    "lat", "long", "merch_lat", "merch_long",
    "hour", "dayofweek", "month",
] + card_feats
FEATURES = categorical_cols + numerical_cols

print(len(FEATURES), "features")
print("NaNs in features:", int(df[FEATURES].isna().sum().sum()))
display(df[FEATURES + [LABEL_COL]].head())

/usr/local/lib/python3.13/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


16 features
NaNs in features: 0


,category,gender,state,log_amt,distance_km,age,log_city_pop,lat,long,merch_lat,merch_long,hour,dayofweek,month,log_secs_since_prev,log_amt_vs_card_mean,is_fraud
0,misc_net,F,NC,1.786747,78.597568,30.814511,8.159375,36.0788,-81.1781,36.011293,-82.048315,0,1,1,9.661607,0.0,0
1,grocery_pos,F,WA,4.684259,30.212176,40.531143,5.010635,48.8878,-118.2105,49.159047,-118.186462,0,1,1,9.661607,0.0,0
2,entertainment,M,ID,5.398660,108.206083,56.950034,8.332068,42.1808,-112.2620,43.150704,-112.154481,0,1,1,9.661607,0.0,0
3,gas_transport,M,MT,3.828641,95.673231,51.969884,7.570443,46.2306,-112.1138,47.034331,-112.561071,0,1,1,9.661607,0.0,0
4,misc_pos,M,VA,3.760269,77.556744,32.763860,4.605170,38.4207,-79.4629,38.674999,-78.632459,0,1,1,9.661607,0.0,0


## 3. Train, validation, and test split

In [ ]:
is_train = df["split"] == "train"
is_test = df["split"] == "test"
is_fraud = df[LABEL_COL] == 1

train_normals = df[is_train & ~is_fraud].sample(N_TRAIN_NORMAL, random_state=SEED)
train_df, val_df = train_test_split(train_normals, test_size=VAL_FRAC, random_state=SEED)
train_fraud = df[is_train & is_fraud]

test_fraud = df[is_test & is_fraud]
test_normals = df[is_test & ~is_fraud].sample(N_TEST_NORMAL, random_state=SEED)
test_eval = pd.concat([test_normals, test_fraud]).sample(frac=1, random_state=SEED)
y_test = test_eval[LABEL_COL].values

print("train normals:", len(train_df))
print("validation normals:", len(val_df))
print("train fraud context pool:", len(train_fraud))
print("test evaluation rows:", len(test_eval))
print("test fraud:", int(y_test.sum()))
print("test fraud prevalence:", round(y_test.mean(), 4))

In [ ]:
prep = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32, min_frequency=20), categorical_cols),
])

X_train = prep.fit_transform(train_df[FEATURES]).astype(np.float32)
X_val = prep.transform(val_df[FEATURES]).astype(np.float32)
X_test = prep.transform(test_eval[FEATURES]).astype(np.float32)

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_val.shape)
print("Test matrix:", X_test.shape)

In [ ]:
results, scores_store = {}, {}

def compute_metrics(y_true, scores, threshold):
    y_pred = (scores > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "true_positives": int(tp),
        "false_positives": int(fp),
        "true_negatives": int(tn),
        "false_negatives": int(fn),
    }

def report(name, y_true, scores, threshold):
    metrics = compute_metrics(y_true, scores, threshold)
    results[name] = metrics
    scores_store[name] = (scores, threshold)
    for key, value in metrics.items():
        print(f"{key:<20}{value:.4f}" if isinstance(value, float) else f"{key:<20}{value}")
    print("\nPrevalence (AP baseline):", round(float(np.mean(y_true)), 4))
    print(classification_report(y_true, (scores > threshold).astype(int), zero_division=0))
    return metrics

## 4. Isolation Forest

In [ ]:
mlflow.set_experiment("Isolation Forest")

with mlflow.start_run():
    iso = IsolationForest(
        n_estimators=300,
        max_samples="auto",
        contamination="auto",
        random_state=SEED,
        n_jobs=-1,
    )
    iso.fit(X_train)

    val_scores = -iso.decision_function(X_val)
    test_scores = -iso.decision_function(X_test)
    threshold = float(np.quantile(val_scores, 1 - FPR_TARGET))
    metrics = report("Isolation Forest", y_test, test_scores, threshold)

    mlflow.log_params({
        "n_estimators": 300,
        "max_samples": "auto",
        "contamination": "auto",
        "random_state": SEED,
        "training_data": "normal_only",
        "threshold_calibration": f"normal_validation_{int((1 - FPR_TARGET) * 100)}th_percentile",
        "card_features": ADD_CARD_FEATURES,
    })
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(
        sk_model=iso,
        name="isolation_forest",
        skops_trusted_types=["sklearn.tree._tree.Tree"],
    )

## 5. Autoencoder

In [ ]:
mlflow.set_experiment("Autoencoder")

with mlflow.start_run():
    n_features = X_train.shape[1]
    ae = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(n_features, activation="linear"),
    ])
    ae.compile(optimizer="adam", loss="mse", metrics=["mae"])
    ae.summary()

    history = ae.fit(
        X_train,
        X_train,
        validation_data=(X_val, X_val),
        epochs=EPOCHS,
        batch_size=256,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
    )
    train_loss = history.history["loss"][-1]
    val_loss = history.history["val_loss"][-1]

    val_scores = np.mean(np.square(X_val - ae.predict(X_val, verbose=0)), axis=1)
    test_scores = np.mean(np.square(X_test - ae.predict(X_test, verbose=0)), axis=1)
    threshold = float(np.quantile(val_scores, 1 - FPR_TARGET))
    metrics = report("Autoencoder", y_test, test_scores, threshold)

    mlflow.log_params({
        "architecture": "64-32-16-32-64",
        "optimizer": "adam",
        "loss": "mse",
        "epochs": EPOCHS,
        "batch_size": 256,
        "early_stopping_patience": 5,
        "random_state": SEED,
        "training_data": "normal_only",
        "threshold_calibration": f"normal_validation_{int((1 - FPR_TARGET) * 100)}th_percentile",
        "card_features": ADD_CARD_FEATURES,
    })
    mlflow.log_metrics({**metrics, "final_train_loss": float(train_loss), "final_val_loss": float(val_loss)})
    mlflow.keras.log_model(ae, name="autoencoder")

## 6. TabPFN few-shot experiments

In [ ]:
os.environ["TABPFN_TOKEN"] = getpass("Enter TabPFN token: ")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("TabPFN device:", TABPFN_DEVICE)

In [ ]:
def tabpfn_frame(data):
    cats = pd.DataFrame(
        ord_enc.transform(data[categorical_cols]),
        columns=categorical_cols,
        index=data.index,
    )
    return pd.concat([cats, data[numerical_cols]], axis=1)

def proba_batched(clf, features, batch_size=5000):
    return np.concatenate([
        clf.predict_proba(features.iloc[i:i + batch_size])[:, 1]
        for i in range(0, len(features), batch_size)
    ])

ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
ord_enc.fit(train_df[categorical_cols])

def run_tabpfn_experiment(k_fraud):
    context = pd.concat([
        train_fraud.sample(k_fraud, random_state=SEED),
        train_df.sample(K_NORMAL, random_state=SEED),
    ])
    X_context = tabpfn_frame(context)
    y_context = context[LABEL_COL].values
    experiment_name = f"TabPFN_{k_fraud}-shot"
    result_name = f"TabPFN_{k_fraud}-shot"

    mlflow.set_experiment(experiment_name)
    with mlflow.start_run():
        clf = TabPFNClassifier(
            categorical_features_indices=list(range(len(categorical_cols))),
            random_state=SEED,
            device=TABPFN_DEVICE,
        )
        clf.fit(X_context, y_context)

        val_sub = val_df.sample(min(5000, len(val_df)), random_state=SEED)
        val_scores = proba_batched(clf, tabpfn_frame(val_sub))
        test_scores = proba_batched(clf, tabpfn_frame(test_eval))
        threshold = float(np.quantile(val_scores, 1 - FPR_TARGET))
        metrics = report(result_name, y_test, test_scores, threshold)

        mlflow.log_params({
            "k_fraud": k_fraud,
            "k_normal": K_NORMAL,
            "random_state": SEED,
            "device": TABPFN_DEVICE,
            "threshold_calibration": f"normal_validation_{int((1 - FPR_TARGET) * 100)}th_percentile",
            "card_features": ADD_CARD_FEATURES,
        })
        mlflow.log_metrics(metrics)

    return metrics

In [ ]:
run_tabpfn_experiment(5)

In [ ]:
run_tabpfn_experiment(10)

In [ ]:
run_tabpfn_experiment(15)

## 7. Model comparison

In [ ]:
summary = pd.DataFrame(results).T[
    ["precision", "recall", "f1", "roc_auc", "average_precision", "false_positive_rate"]
]
print(f"Test prevalence = {y_test.mean():.4f} (average-precision baseline), FPR target = {FPR_TARGET}")
display(summary.round(4))

In [ ]:
fraud_mask = y_test == 1
fraud_categories = test_eval["category"].values[fraud_mask]

by_category = pd.DataFrame({
    name: pd.Series((scores > threshold)[fraud_mask]).groupby(fraud_categories).mean()
    for name, (scores, threshold) in scores_store.items()
})
by_category.insert(0, "n_fraud", pd.Series(fraud_categories).value_counts())
display(by_category.sort_values("n_fraud", ascending=False).round(3))